# Day 2 — Data Understanding, Validation & Cleaning
## HR Alto Network Employee Attrition Project

**Goal:** Validate the original dataset, investigate quality issues, document decisions, and save a separate cleaned dataset.

## Workflow
Load Original Data → Missing Values → Duplicates → Data Types → Invalid Values → Logical Consistency → Categorical Consistency → Outliers → Constant Columns → Cleaning Decisions → Final Validation → Data Quality Report → Save Cleaned Dataset

# 1. Load Original Data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

candidate_paths = [
    Path("../data/raw/Palo Alto Networks.csv"),
    Path("../data/raw/Palo Alto Networks(1).csv"),
    Path("../data/Palo Alto Networks.csv"),
    Path("../data/Palo Alto Networks(1).csv"),
    Path("Palo Alto Networks.csv"),
    Path("Palo Alto Networks(1).csv")
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Original dataset not found. Update candidate_paths.")
df_original = pd.read_csv(data_path)
df = df_original.copy()
print("Original Shape:", df_original.shape)

Original Shape: (1470, 31)


# 2. Missing Values

In [2]:
missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isna().sum().values
})
missing_summary["Missing_Percentage"] = (missing_summary["Missing_Count"]/len(df)*100).round(2)
display(missing_summary)
print("Total Missing Values:", int(df.isna().sum().sum()))

,Column,Missing_Count,Missing_Percentage
0,Age,0,0.0
1,Attrition,0,0.0
2,BusinessTravel,0,0.0
3,DailyRate,0,0.0
4,Department,0,0.0
5,DistanceFromHome,0,0.0
6,Education,0,0.0
7,EducationField,0,0.0
8,EnvironmentSatisfaction,0,0.0
9,Gender,0,0.0


Total Missing Values: 0


# 3. Duplicate Check

In [3]:
duplicate_count = int(df.duplicated().sum())
print("Duplicate Rows:", duplicate_count)
if duplicate_count:
    df = df.drop_duplicates().copy()
    print("Duplicates removed.")
else:
    print("No duplicate rows found.")

Duplicate Rows: 0
No duplicate rows found.


# 4. Data Type Validation

In [4]:
dtype_validation = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Unique_Count": df.nunique(dropna=False).values
})
display(dtype_validation)

,Column,Data_Type,Unique_Count
0,Age,int64,43
1,Attrition,int64,2
2,BusinessTravel,object,3
3,DailyRate,int64,886
4,Department,object,3
5,DistanceFromHome,int64,29
6,Education,int64,5
7,EducationField,object,6
8,EnvironmentSatisfaction,int64,4
9,Gender,object,2


# 5. Invalid Value Check

In [5]:
non_negative_columns = [
    "Age","DailyRate","HourlyRate","MonthlyIncome","MonthlyRate",
    "DistanceFromHome","TotalWorkingYears","NumCompaniesWorked",
    "YearsAtCompany","YearsInCurrentRole","YearsSinceLastPromotion",
    "YearsWithCurrManager","TrainingTimesLastYear","PercentSalaryHike"
]
invalid_results = []
for col in non_negative_columns:
    if col in df.columns:
        invalid_results.append({"Column":col,"Invalid_Negative_Count":int((df[col]<0).sum())})
invalid_value_summary = pd.DataFrame(invalid_results)
display(invalid_value_summary)

,Column,Invalid_Negative_Count
0,Age,0
1,DailyRate,0
2,HourlyRate,0
3,MonthlyIncome,0
4,MonthlyRate,0
5,DistanceFromHome,0
6,TotalWorkingYears,0
7,NumCompaniesWorked,0
8,YearsAtCompany,0
9,YearsInCurrentRole,0


## 5.1 Bounded Rating / Code Validation

In [6]:
range_rules = {
    "Attrition":(0,1),"Education":(1,5),"EnvironmentSatisfaction":(1,4),
    "JobInvolvement":(1,4),"JobLevel":(1,5),"JobSatisfaction":(1,4),
    "PerformanceRating":(1,4),"RelationshipSatisfaction":(1,4),
    "StockOptionLevel":(0,3),"WorkLifeBalance":(1,4)
}
range_results = []
for col,(low,high) in range_rules.items():
    if col in df.columns:
        range_results.append({
            "Column":col,"Lower_Bound":low,"Upper_Bound":high,
            "Invalid_Record_Count":int(((df[col]<low)|(df[col]>high)).sum())
        })
range_validation = pd.DataFrame(range_results)
display(range_validation)

,Column,Lower_Bound,Upper_Bound,Invalid_Record_Count
0,Attrition,0,1,0
1,Education,1,5,0
2,EnvironmentSatisfaction,1,4,0
3,JobInvolvement,1,4,0
4,JobLevel,1,5,0
5,JobSatisfaction,1,4,0
6,PerformanceRating,1,4,0
7,RelationshipSatisfaction,1,4,0
8,StockOptionLevel,0,3,0
9,WorkLifeBalance,1,4,0


# 6. Logical Consistency

In [7]:
logical_rules = {
    "YearsAtCompany > TotalWorkingYears": df["YearsAtCompany"] > df["TotalWorkingYears"],
    "YearsInCurrentRole > YearsAtCompany": df["YearsInCurrentRole"] > df["YearsAtCompany"],
    "YearsSinceLastPromotion > YearsAtCompany": df["YearsSinceLastPromotion"] > df["YearsAtCompany"],
    "YearsWithCurrManager > YearsAtCompany": df["YearsWithCurrManager"] > df["YearsAtCompany"]
}
logical_validation = pd.DataFrame({
    "Validation_Rule":list(logical_rules.keys()),
    "Invalid_Record_Count":[int(x.sum()) for x in logical_rules.values()]
})
display(logical_validation)

logical_issue_rows = df[logical_rules["YearsAtCompany > TotalWorkingYears"]]
print("Rows flagged:", len(logical_issue_rows))
display(logical_issue_rows[[
    "Age","TotalWorkingYears","YearsAtCompany",
    "YearsInCurrentRole","YearsSinceLastPromotion","YearsWithCurrManager"
]].head(20))

,Validation_Rule,Invalid_Record_Count
0,YearsAtCompany > TotalWorkingYears,0
1,YearsInCurrentRole > YearsAtCompany,0
2,YearsSinceLastPromotion > YearsAtCompany,0
3,YearsWithCurrManager > YearsAtCompany,0


Rows flagged: 0


,Age,TotalWorkingYears,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager


# 7. Categorical Consistency

In [8]:
categorical_columns = df.select_dtypes(include=["object","category"]).columns.tolist()
categorical_results = []
for col in categorical_columns:
    original_unique = int(df[col].dropna().nunique())
    normalized_unique = int(df[col].astype("string").str.strip().str.lower().dropna().nunique())
    categorical_results.append({
        "Column":col,"Original_Unique":original_unique,
        "Normalized_Unique":normalized_unique,
        "Possible_Inconsistency":original_unique != normalized_unique
    })
categorical_consistency = pd.DataFrame(categorical_results)
display(categorical_consistency)

,Column,Original_Unique,Normalized_Unique,Possible_Inconsistency
0,BusinessTravel,3,3,False
1,Department,3,3,False
2,EducationField,6,6,False
3,Gender,2,2,False
4,JobRole,9,9,False
5,MaritalStatus,3,3,False
6,OverTime,2,2,False


# 8. Outlier Analysis

In [9]:
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
outlier_columns = [c for c in numerical_columns if c != "Attrition"]
outlier_results = []
for col in outlier_columns:
    q1,q3 = df[col].quantile([0.25,0.75])
    iqr = q3-q1
    lower,upper = q1-1.5*iqr,q3+1.5*iqr
    mask = (df[col]<lower)|(df[col]>upper)
    outlier_results.append({
        "Column":col,"Lower_Bound":round(lower,2),
        "Upper_Bound":round(upper,2),"Outlier_Count":int(mask.sum()),
        "Outlier_Percentage":round(mask.mean()*100,2)
    })
outlier_summary = pd.DataFrame(outlier_results).sort_values(
    "Outlier_Count",ascending=False).reset_index(drop=True)
display(outlier_summary)

,Column,Lower_Bound,Upper_Bound,Outlier_Count,Outlier_Percentage
0,TrainingTimesLastYear,0.50,4.50,238,16.19
1,PerformanceRating,3.00,3.00,226,15.37
2,MonthlyIncome,-5291.00,16581.00,114,7.76
3,YearsSinceLastPromotion,-4.50,7.50,107,7.28
4,YearsAtCompany,-6.00,18.00,104,7.07
5,StockOptionLevel,-1.50,2.50,85,5.78
6,TotalWorkingYears,-7.50,28.50,63,4.29
7,NumCompaniesWorked,-3.50,8.50,52,3.54
8,YearsInCurrentRole,-5.50,14.50,21,1.43
9,YearsWithCurrManager,-5.50,14.50,14,0.95


# 9. Constant / Unnecessary Columns

In [10]:
constant_columns = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print("Constant Columns:", constant_columns if constant_columns else "None")

Constant Columns: None


# 10. Cleaning Decisions

- Missing values: no treatment when total is 0.
- Duplicates: remove only true duplicate rows.
- Invalid values: flag and investigate.
- `YearsAtCompany > TotalWorkingYears`: retain and document because no reliable correction rule is available.
- Categorical consistency: no change when normalized unique counts match.
- IQR outliers: do not remove rows based only on IQR.
- Constant columns: remove only confirmed constant columns.

In [11]:
df_cleaned = df.copy()
if constant_columns:
    df_cleaned = df_cleaned.drop(columns=constant_columns)
print("Original Shape:", df_original.shape)
print("Final Cleaned Shape:", df_cleaned.shape)

Original Shape: (1470, 31)
Final Cleaned Shape: (1470, 31)


# 11. Final Validation

In [12]:
final_missing = int(df_cleaned.isna().sum().sum())
final_duplicates = int(df_cleaned.duplicated().sum())
final_logical_validation = pd.DataFrame({
    "Validation_Rule": list(logical_rules.keys()),
    "Invalid_Record_Count": [
        int((df_cleaned["YearsAtCompany"] > df_cleaned["TotalWorkingYears"]).sum()),
        int((df_cleaned["YearsInCurrentRole"] > df_cleaned["YearsAtCompany"]).sum()),
        int((df_cleaned["YearsSinceLastPromotion"] > df_cleaned["YearsAtCompany"]).sum()),
        int((df_cleaned["YearsWithCurrManager"] > df_cleaned["YearsAtCompany"]).sum())
    ]
})
print("Missing:", final_missing, "| Duplicates:", final_duplicates)
display(final_logical_validation)
assert final_missing == 0
assert final_duplicates == 0

Missing: 0 | Duplicates: 0


,Validation_Rule,Invalid_Record_Count
0,YearsAtCompany > TotalWorkingYears,0
1,YearsInCurrentRole > YearsAtCompany,0
2,YearsSinceLastPromotion > YearsAtCompany,0
3,YearsWithCurrManager > YearsAtCompany,0


# 12. Data Quality Report & Automatic Saving

In [13]:
output_dir = Path("../outputs/day_2")
tables_dir = output_dir / "tables"
reports_dir = output_dir / "reports"
cleaned_dir = Path("../data/cleaned")
for d in [tables_dir,reports_dir,cleaned_dir]:
    d.mkdir(parents=True,exist_ok=True)

cleaned_path = cleaned_dir / "Palo Alto Networks_cleaned.csv"
df_cleaned.to_csv(cleaned_path,index=False)

tables = {
    "missing_values":missing_summary,
    "data_type_validation":dtype_validation,
    "invalid_value_summary":invalid_value_summary,
    "range_validation":range_validation,
    "logical_validation":logical_validation,
    "categorical_consistency":categorical_consistency,
    "outlier_summary":outlier_summary
}
for name,table in tables.items():
    table.to_csv(tables_dir/f"{name}.csv",index=False)

data_quality_summary = pd.DataFrame({
    "Check":["Original Shape","Final Shape","Missing Values","Duplicate Rows",
             "Constant Columns","IQR Outlier Rows Removed",
             "YearsAtCompany > TotalWorkingYears"],
    "Result":[
        str(df_original.shape),str(df_cleaned.shape),final_missing,final_duplicates,
        ", ".join(constant_columns) if constant_columns else "None",0,
        int((df_cleaned["YearsAtCompany"] > df_cleaned["TotalWorkingYears"]).sum())
    ]
})
data_quality_summary.to_csv(tables_dir/"data_quality_summary.csv",index=False)

report = (
    "# Day 2 Data Quality Report\n\n"
    f"Original shape: {df_original.shape}\n\n"
    f"Final shape: {df_cleaned.shape}\n\n"
    f"Missing values: {final_missing}\n\n"
    f"Duplicate rows: {final_duplicates}\n\n"
    "IQR outlier rows removed: 0\n\n"
    f"YearsAtCompany > TotalWorkingYears: {int((df_cleaned['YearsAtCompany'] > df_cleaned['TotalWorkingYears']).sum())}\n\n"
    "Logical inconsistencies were documented and retained."
)
(reports_dir/"day_2_data_quality_report.md").write_text(report,encoding="utf-8")
print("Cleaned dataset:",cleaned_path)
print("Tables:",tables_dir)
print("Report:",reports_dir/"day_2_data_quality_report.md")

Cleaned dataset: ../data/cleaned/Palo Alto Networks_cleaned.csv
Tables: ../outputs/day_2/tables
Report: ../outputs/day_2/reports/day_2_data_quality_report.md
